# ZARX-1B Training Notebook (Google Colab)
1B parameter code + ARC reasoning model trained from scratch.

**GPU:** T4 (16GB VRAM) | **Runtime:** 4-12hrs per session

## Setup
1. Go to Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. Training auto-resumes from checkpoints

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch>=2.1.0 bitsandbytes wandb huggingface_hub
!pip install -q datasets tokenizers accelerate datasketch
!pip install -q flash-attn --no-build-isolation
print('Dependencies installed!')

In [ ]:
#@title 2. Mount Google Drive & Login
from google.colab import drive
drive.mount('/content/drive')

import os

# HuggingFace login (set your token below or use Colab secrets)
HF_TOKEN = '' #@param {type:"string"}
if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}

# W&B login
WANDB_KEY = '' #@param {type:"string"}
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)

print('Drive mounted and logins complete!')

In [ ]:
#@title 3. Clone ZARX-1B Repository
%cd /content
!rm -rf ZARX-1B  # Clean if re-running
!git clone https://github.com/YOURUSERNAME/ZARX-1B.git
%cd ZARX-1B
!pip install -e .
print('Repository cloned!')

In [ ]:
#@title 4. Verify GPU & Environment
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')

# Test model fits in memory
import sys
sys.path.insert(0, '.')
from src.model import build_model
model = build_model('configs/model_config.json')
model = model.cuda()
x = torch.randint(0, 32000, (1, 128)).cuda()
out = model(x)
print(f'Forward pass OK! Output shape: {out["logits"].shape}')
print(f'GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB used')
del model, x, out
torch.cuda.empty_cache()

In [ ]:
#@title 5. Download Data (Run once)
# Download ProX code data
!python scripts/download_data.py --download_prox --max_prox_examples 2000000 --output_dir /content/data/raw

# Download ARC-AGI data
!python scripts/download_data.py --download_arc --output_dir /content/data/raw

In [ ]:
#@title 6. Generate ARC Augmented Data
!python src/arc_augment.py \
    --arc1_path /content/data/raw/arc/arc-agi-1/training \
    --arc2_path /content/data/raw/arc/arc-agi-2/data/training \
    --variations_per_task 500 \
    --output_path /content/data/raw/arc/augmented_arc.jsonl \
    --format code_reasoning

In [ ]:
#@title 7. Preprocess Data
# Process ProX data
!python src/data_pipeline.py --mode process \
    --input_file /content/data/raw/prox/prox-shard-0000.jsonl \
    --output_file /content/data/processed/prox_processed.jsonl \
    --source prox

# Process ARC data
!python src/data_pipeline.py --mode process \
    --input_file /content/data/raw/arc/augmented_arc.jsonl \
    --output_file /content/data/processed/arc_processed.jsonl \
    --source arc

# Merge
!python src/data_pipeline.py --mode merge \
    --processed_dir /content/data/processed \
    --output_file /content/data/processed/zarx_pretrain.jsonl

In [ ]:
#@title 8. Train Tokenizer (Run once)
!python scripts/train_tokenizer.py \
    --data_path /content/data/processed \
    --output_path /content/configs/tokenizer.json \
    --vocab_size 32000 \
    --max_lines 5000000

In [ ]:
#@title 9. Start Training
# Configuration
HF_REPO = '' #@param {type:"string"}  # Your HuggingFace checkpoint repo
HF_TOKEN = '' #@param {type:"string"}  # Your HuggingFace token

!python src/train.py \
    --model_config configs/model_config.json \
    --tokenizer_path /content/configs/tokenizer.json \
    --data_path /content/data/processed \
    --micro_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 3e-4 \
    --warmup_steps 2000 \
    --total_tokens 10000000000 \
    --use_8bit_adam \
    --checkpoint_dir /content/checkpoints \
    --drive_dir /content/drive/MyDrive/ZARX-1B \
    --hf_repo_id {HF_REPO} \
    --hf_token {HF_TOKEN} \
    --save_every_local 100 \
    --save_every_drive 500 \
    --save_every_hf 1000 \
    --log_every_steps 10 \
    --wandb_project zarx-1b \
    --output_dir /content/zarx-1b-final

In [ ]:
#@title 10. Quick Evaluation (After training)
import sys
sys.path.insert(0, '.')
from src.model import ZARXModel, ZARXConfig
from src.eval import quick_eval
from tokenizers import Tokenizer

# Load model
config = ZARXConfig.from_json('configs/model_config.json')
model = ZARXModel(config)
model.tie_weights()

# Load checkpoint
import torch
ckpt = torch.load('/content/checkpoints/checkpoint-latest.pt', map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model = model.cuda()

# Load tokenizer
tokenizer = Tokenizer.from_file('/content/configs/tokenizer.json')

# Quick eval
quick_eval(model, tokenizer, device='cuda')